# Maskera v20 Privacy-Clean GPU Sweep (Google Colab Ultra)

Runs the complete deterministic, privacy-attested v20 training and distillation sweep on an A100 or L4 GPU with checkpoints synced to Google Drive:
1. **v20 synthetic-only data** (120k base + balanced replay + hard negatives + clinical/eponym contrast + homograph contrast + LinkedIn/freetext/ALL-CAPS/region precision).
2. **3-seed KB-BERT teacher ensemble** (`1337`, `2024`, `42`) with `bf16` and cosine schedule.
3. **Multi-candidate distillation sweep** from the ensemble: 3 full-size (~43 MB q4) seeds, hidden-state cosine distillation, and low-rank embedding students (`rank=256`, `rank=192`).
4. **Full JS evaluation battery** (all release gates + freetext + homograph + clinical retention) next to the shipped v19 baseline.
5. **Separate one-shot blind eval cell** at the bottom (Gemini API) to score the winning candidate ONCE on untouched phrasing.

In [ ]:
# 1. Check GPU and mount Google Drive for persistent checkpoints
import os, torch
from google.colab import drive

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
assert torch.cuda.is_available(), 'No CUDA GPU attached. Switch Runtime -> Change runtime type -> A100 or L4.'
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}, bf16={torch.cuda.is_bf16_supported()}')

drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/maskera-v20b'
os.makedirs(DRIVE_OUT, exist_ok=True)
os.environ['DRIVE_OUT'] = DRIVE_OUT
print('Drive output directory:', DRIVE_OUT)

In [ ]:
%%bash
# 2. Load Maskera checkout: prefers Drive tarball (preserves local uncommitted files), else clones main
set -e
WORKDIR=/content/maskera
TARBALL="$DRIVE_OUT/maskera-src.tar.gz"

rm -rf "$WORKDIR"
if [ -f "$TARBALL" ]; then
  echo "Extracting local snapshot from $TARBALL..."
  mkdir -p "$WORKDIR"
  tar -xzf "$TARBALL" -C "$WORKDIR"
else
  echo "Cloning https://github.com/joelhagvall/maskera.git ..."
  git clone --depth 1 https://github.com/joelhagvall/maskera.git "$WORKDIR"
fi
ls -la "$WORKDIR/training"

In [ ]:
%%bash
# 3. Install Node 22, pnpm 10.33.0, Python NLP/ONNX deps (keeps Colab's CUDA PyTorch), and download v19 baseline
set -e
if ! node -v 2>/dev/null | grep -q '^v22\.'; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
  apt-get install -y -qq nodejs
fi
npm install -g pnpm@10.33.0
export COREPACK_ENABLE_DOWNLOAD_PROMPT=0

pip install -q --upgrade \
  'transformers>=4.57,<5' 'datasets>=3.2' 'seqeval>=1.2' 'accelerate>=1.0' \
  'optimum>=2.0' 'optimum-onnx>=0.1' 'onnx>=1.17' 'onnx-ir>=0.1' 'onnxruntime>=1.20' 'huggingface_hub>=0.30'
pip uninstall -y -q diffusers || true

cd /content/maskera
pnpm install --frozen-lockfile
pnpm --filter @maskera/core build
pnpm --filter maskera build

# Fetch shipped v19 q4 artifact so the sweep table compares directly against live v19
python3 - <<'PY'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='joelhagvall/maskera-sv-ner',
    revision='7a0063375d1baabf66cf9a357dad5f46aea7008e',
    local_dir='/content/maskera/training/student-v19-privacy-precision2-onnx',
    allow_patterns=['config.json', 'tokenizer*', 'special_tokens_map.json', 'vocab.txt', 'onnx/model_q4.onnx'],
)
print('Downloaded v19 baseline.')
PY

In [ ]:
%%bash
# 4. Run the complete privacy-clean v20 GPU sweep (resumes any completed teachers/students from Drive)
# Tip: on Google AI Ultra you can enable Runtime -> Manage sessions -> Background execution and close the tab.
set -e
cd /content/maskera
PY=python3 DRIVE_OUT="$DRIVE_OUT" sh training/run_v20_sweep.sh

In [ ]:
# 5. Print the saved sweep comparison report
import json
report_path = f"{os.environ['DRIVE_OUT']}/v20-sweep-report.json"
with open(report_path) as f:
    report = json.load(f)
for r in report['results']:
    print(
        f"{r['model']:36s} | {r['mb']:5.1f} MB | gates={'PASS' if r['releaseGatesPass'] else 'FAIL'} "
        f"| cur={r['curated']['spanF1']}/{r['curated']['leaks']} "
        f"| adr={r['adr']['spanF1']}/{r['adr']['leaks']} "
        f"| rare={r['rareSurnames']['maskedPct']}%({r['rareSurnames']['leaks']}) "
        f"| LI={r['linkedin']['precision']}/{r['linkedin']['spanF1']} "
        f"| FT={r['freetext']['precision']}/{r['freetext']['spanF1']} "
        f"| homo={r['homograph']['homographLeakPct']}%({r['homograph']['wordFp']}) "
        f"| clin={r['clinical']['cleanTexts']}/{r['clinical']['totalTexts'] if r['clinical'] else '-'}"
    )

## 6. One-shot untouched blind evaluation (run ONCE on your chosen winner)

Add `GEMINI_API_KEY` in Colab **Secrets** (the key icon in the left sidebar) and set `WINNER_MODEL` below to the single best candidate from step 5. Never tune on this set.

In [ ]:
import os, subprocess
from google.colab import userdata

WINNER_MODEL = 'student-v20b-s1337-onnx'  # Replace with the winning candidate from step 5
BLIND_FILE = f"{os.environ['DRIVE_OUT']}/corpus-blind-v20.mjs"

env = os.environ.copy()
env['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

if not os.path.exists(BLIND_FILE):
    subprocess.run([
        'node', '/content/maskera/training/gemini_synth_helper.mjs',
        '--mode', 'blind-eval', '--per-category', '20', '--out', BLIND_FILE,
    ], env=env, check=True)

subprocess.run([
    'node', '/content/maskera/training/grade_candidate.mjs',
    '--blind', BLIND_FILE,
    '--out', f"{os.environ['DRIVE_OUT']}/v20-blind-verdict.json",
    'student-v19-privacy-precision2-onnx',
    WINNER_MODEL,
], cwd='/content/maskera', check=True)